# Error Analysis: Internal vs External Benchmark Transfer Gap

Analyzes per-question eval results from `--error-log` to diagnose why
internal MC accuracy is high but external benchmarks remain near random.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

ERROR_DIR = Path(r'D:/hist_LLM/periods/1900_1949/error_analysis')
STAGES = ['mid_final', 'sft_final']

def load_details(stage, task):
    path = ERROR_DIR / stage / f'{task}_details.jsonl'
    if not path.exists():
        return pd.DataFrame()
    records = [json.loads(line) for line in open(path, encoding='utf-8')]
    df = pd.DataFrame(records)
    df['task'] = task
    df['stage'] = stage
    return df

# Load all tasks for both stages
TASKS = [
    'ARC-Challenge', 'HellaSwag', 'RACE-Middle', 'RACE-High',
    'Winogrande', 'PIQA', 'GSM-MC',
    'InternalMC_A', 'InternalMC_B', 'InternalMC_C',
    'InternalMC_D', 'InternalMC_E', 'InternalMC_F',
    'LAB', 'LAB2000',
]

dfs = {}
for stage in STAGES:
    frames = [load_details(stage, t) for t in TASKS]
    dfs[stage] = pd.concat([f for f in frames if len(f) > 0], ignore_index=True)
    print(f'{stage}: {len(dfs[stage])} total questions across {dfs[stage]["task"].nunique()} tasks')

## 1. Accuracy Summary: Internal vs External

In [ ]:
RANDOM_BASELINES = {
    'ARC-Challenge': 0.25, 'HellaSwag': 0.25, 'RACE-Middle': 0.25, 'RACE-High': 0.25,
    'Winogrande': 0.50, 'PIQA': 0.50, 'GSM-MC': 0.25,
    'InternalMC_A': 0.25, 'InternalMC_B': 0.25, 'InternalMC_C': 0.25,
    'InternalMC_D': 0.25, 'InternalMC_E': 0.25, 'InternalMC_F': 0.25,
    'LAB': 0.25, 'LAB2000': 0.25,
}

for stage in STAGES:
    df = dfs[stage]
    print(f'\n=== {stage.upper()} ===')
    summary = df.groupby('task')['correct'].agg(['mean', 'sum', 'count'])
    summary.columns = ['accuracy', 'correct', 'total']
    summary['baseline'] = summary.index.map(RANDOM_BASELINES)
    summary['vs_random'] = summary['accuracy'] - summary['baseline']
    summary = summary.sort_values('vs_random', ascending=False)
    
    # Color formatting
    for _, row in summary.iterrows():
        marker = '+' if row['vs_random'] > 0.02 else ('~' if abs(row['vs_random']) <= 0.02 else '-')
        print(f"  {marker} {row.name:20s}  {row['accuracy']:.1%}  ({int(row['correct'])}/{int(row['total'])})  vs_random: {row['vs_random']:+.1%}")

## 2. Answer Position Bias

Does the model systematically prefer certain answer positions (A, B, C, D)?
A well-calibrated model should predict each letter roughly uniformly.

In [ ]:
stage = 'mid_final'
df = dfs[stage]

# Separate internal vs external
internal_tasks = [t for t in TASKS if t.startswith('InternalMC')]
external_tasks = ['ARC-Challenge', 'HellaSwag', 'RACE-Middle', 'RACE-High', 'GSM-MC']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, tasks) in zip(axes, [('External Benchmarks', external_tasks), ('Internal MC (A-F)', internal_tasks)]):
    subset = df[df['task'].isin(tasks)]
    
    # Predicted distribution
    pred_counts = subset['predicted'].value_counts().sort_index()
    expected_counts = subset['expected'].value_counts().sort_index()
    
    # Align indices
    all_letters = sorted(set(pred_counts.index) | set(expected_counts.index))
    pred_vals = [pred_counts.get(l, 0) for l in all_letters]
    exp_vals = [expected_counts.get(l, 0) for l in all_letters]
    
    x = np.arange(len(all_letters))
    w = 0.35
    ax.bar(x - w/2, pred_vals, w, label='Predicted', color='#e74c3c', alpha=0.8)
    ax.bar(x + w/2, exp_vals, w, label='Expected (ground truth)', color='#2ecc71', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(all_letters)
    ax.set_xlabel('Answer Letter')
    ax.set_ylabel('Count')
    ax.set_title(f'{label} — {stage}')
    ax.legend()

plt.tight_layout()
plt.show()

# Print exact numbers
for label, tasks in [('External', external_tasks), ('Internal', internal_tasks)]:
    subset = df[df['task'].isin(tasks)]
    pred_dist = subset['predicted'].value_counts(normalize=True).sort_index()
    exp_dist = subset['expected'].value_counts(normalize=True).sort_index()
    print(f'\n{label} predicted distribution: {dict(pred_dist.round(3))}')
    print(f'{label} expected distribution:  {dict(exp_dist.round(3))}')

## 3. Per-Task Answer Position Bias

Breakdown per task to see which benchmarks have the worst positional bias.

In [ ]:
stage = 'mid_final'
df = dfs[stage]

print(f'Per-task predicted letter distribution ({stage}):')
print(f'{"Task":25s} {"A":>7s} {"B":>7s} {"C":>7s} {"D":>7s}  {"Most":>6s}  {"Acc":>6s}')
print('-' * 80)

for task in TASKS:
    subset = df[df['task'] == task]
    if len(subset) == 0:
        continue
    pred_dist = subset['predicted'].value_counts(normalize=True)
    acc = subset['correct'].mean()
    most_common = pred_dist.idxmax()
    vals = [f"{pred_dist.get(l, 0):.1%}" for l in ['A', 'B', 'C', 'D']]
    bias_flag = '***' if pred_dist.max() > 0.40 else ''
    print(f"{task:25s} {vals[0]:>7s} {vals[1]:>7s} {vals[2]:>7s} {vals[3]:>7s}  {most_common:>4s}    {acc:.1%} {bias_flag}")

## 4. Confidence Distribution: Internal vs External

How confident is the model when right vs wrong?
- If confidently wrong on external: the model has learned a wrong pattern
- If near-uniform on external: the model hasn't learned anything transferable

In [ ]:
stage = 'mid_final'
df = dfs[stage]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

categories = [
    ('Internal MC — Correct', df[(df['task'].isin(internal_tasks)) & (df['correct'] == True)]),
    ('Internal MC — Wrong', df[(df['task'].isin(internal_tasks)) & (df['correct'] == False)]),
    ('External — Correct', df[(df['task'].isin(external_tasks)) & (df['correct'] == True)]),
    ('External — Wrong', df[(df['task'].isin(external_tasks)) & (df['correct'] == False)]),
]

for ax, (title, subset) in zip(axes.flat, categories):
    if len(subset) == 0:
        ax.set_title(f'{title} (no data)')
        continue
    ax.hist(subset['confidence'], bins=50, range=(0, 1), alpha=0.8, 
            color='#2ecc71' if 'Correct' in title else '#e74c3c')
    ax.set_xlabel('Confidence (max softmax prob)')
    ax.set_ylabel('Count')
    ax.set_title(f'{title} (n={len(subset)}, median={subset["confidence"].median():.2f})')
    ax.axvline(0.25, color='gray', linestyle='--', alpha=0.5, label='random (0.25)')
    ax.legend()

plt.suptitle(f'Confidence Distributions — {stage}', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Per-Task Confidence Breakdown

In [ ]:
stage = 'mid_final'
df = dfs[stage]

print(f'Confidence statistics per task ({stage}):')
print(f'{"Task":25s} {"Acc":>6s}  {"Conf(right)":>11s} {"Conf(wrong)":>11s} {"Diagnosis"}')
print('-' * 90)

for task in TASKS:
    subset = df[df['task'] == task]
    if len(subset) == 0:
        continue
    acc = subset['correct'].mean()
    right = subset[subset['correct'] == True]['confidence']
    wrong = subset[subset['correct'] == False]['confidence']
    
    conf_right = f"{right.median():.2f}" if len(right) > 0 else 'N/A'
    conf_wrong = f"{wrong.median():.2f}" if len(wrong) > 0 else 'N/A'
    
    # Diagnosis
    baseline = RANDOM_BASELINES.get(task, 0.25)
    if acc > baseline + 0.10:
        diagnosis = 'LEARNED'
    elif len(wrong) > 0 and wrong.median() > 0.5:
        diagnosis = 'CONFIDENTLY WRONG'
    elif len(wrong) > 0 and wrong.median() < 0.35:
        diagnosis = 'NEAR-RANDOM'
    else:
        diagnosis = 'WEAK SIGNAL'
    
    print(f"{task:25s} {acc:6.1%}  {conf_right:>11s} {conf_wrong:>11s} {diagnosis}")

## 6. Transfer Gap: Generator → Aligned Benchmark

Each generator was designed to align with a specific external benchmark.
Compare internal MC accuracy to its aligned external benchmark.

In [ ]:
stage = 'mid_final'
df = dfs[stage]

alignment = {
    'InternalMC_A': 'ARC-Challenge',  # Factual QA
    'InternalMC_B': 'ARC-Challenge',  # CoT
    'InternalMC_C': 'RACE-Middle',    # Reading comprehension
    'InternalMC_D': 'GSM-MC',         # Quantitative
    'InternalMC_E': 'HellaSwag',      # Sentence completion
    'InternalMC_F': 'RACE-High',      # Instruction / passage
}

print(f'Generator-to-Benchmark Transfer ({stage}):')
print(f'{"Generator":15s} {"Internal":>10s} {"Aligned Ext":>12s} {"Ext Bench":>15s} {"Gap":>8s}')
print('-' * 65)

for gen, bench in alignment.items():
    gen_acc = df[df['task'] == gen]['correct'].mean() if len(df[df['task'] == gen]) > 0 else float('nan')
    bench_acc = df[df['task'] == bench]['correct'].mean() if len(df[df['task'] == bench]) > 0 else float('nan')
    gap = gen_acc - bench_acc
    print(f"{gen:15s} {gen_acc:10.1%} {bench_acc:12.1%} {bench:>15s} {gap:+8.1%}")

## 7. Mid vs SFT Comparison

Does SFT help or hurt on external benchmarks?

In [ ]:
print(f'{"Task":25s} {"Mid":>8s} {"SFT":>8s} {"Delta":>8s}')
print('-' * 55)

for task in TASKS:
    mid_acc = dfs['mid_final'][dfs['mid_final']['task'] == task]['correct'].mean() if task in dfs['mid_final']['task'].values else float('nan')
    sft_acc = dfs['sft_final'][dfs['sft_final']['task'] == task]['correct'].mean() if task in dfs['sft_final']['task'].values else float('nan')
    delta = sft_acc - mid_acc
    arrow = '+' if delta > 0.01 else ('-' if delta < -0.01 else '~')
    print(f"{task:25s} {mid_acc:8.1%} {sft_acc:8.1%} {delta:+8.1%} {arrow}")

## 8. Sample Wrong Answers: External Benchmarks

Look at high-confidence wrong answers to understand what pattern the model learned.

In [ ]:
stage = 'mid_final'
df = dfs[stage]

for task in external_tasks[:3]:  # ARC, HellaSwag, RACE-Middle
    subset = df[(df['task'] == task) & (df['correct'] == False)]
    subset = subset.sort_values('confidence', ascending=False)
    
    print(f'\n{"=" * 80}')
    print(f'{task}: Top 3 HIGHEST-CONFIDENCE wrong answers')
    print(f'{"=" * 80}')
    
    for _, row in subset.head(3).iterrows():
        print(f'\nConfidence: {row["confidence"]:.2%} | Predicted: {row["predicted"]} | Expected: {row["expected"]}')
        print(f'Probs: {row["probs"]}')
        # Truncate long questions
        q = row['question']
        if len(q) > 400:
            q = q[:400] + '...'
        print(f'Question: {q}')

## 9. Entropy Analysis

Shannon entropy of the probability distribution per question.
- Max entropy for 4 choices = log2(4) = 2.0 (uniform/random)
- Low entropy = model is confident (right or wrong)

In [ ]:
stage = 'mid_final'
df = dfs[stage].copy()

def compute_entropy(probs_dict):
    probs = np.array(list(probs_dict.values()))
    probs = probs[probs > 0]  # avoid log(0)
    return -np.sum(probs * np.log2(probs))

df['entropy'] = df['probs'].apply(compute_entropy)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, tasks) in zip(axes, [('External', external_tasks), ('Internal MC', internal_tasks)]):
    subset = df[df['task'].isin(tasks)]
    right = subset[subset['correct'] == True]['entropy']
    wrong = subset[subset['correct'] == False]['entropy']
    
    ax.hist(right, bins=40, range=(0, 2.1), alpha=0.6, label=f'Correct (n={len(right)})', color='#2ecc71')
    ax.hist(wrong, bins=40, range=(0, 2.1), alpha=0.6, label=f'Wrong (n={len(wrong)})', color='#e74c3c')
    ax.axvline(2.0, color='gray', linestyle='--', alpha=0.5, label='max entropy (uniform)')
    ax.set_xlabel('Shannon Entropy (bits)')
    ax.set_ylabel('Count')
    ax.set_title(f'{label} — {stage}')
    ax.legend()

plt.tight_layout()
plt.show()

## 10. Question Length Analysis

Are longer or shorter questions harder for the model?

In [ ]:
stage = 'mid_final'
df = dfs[stage].copy()
df['q_len'] = df['question'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, tasks) in zip(axes, [('External', external_tasks), ('Internal MC', internal_tasks)]):
    subset = df[df['task'].isin(tasks)]
    
    # Bin by question length quartiles
    subset = subset.copy()
    subset['q_bin'] = pd.qcut(subset['q_len'], q=5, duplicates='drop')
    acc_by_len = subset.groupby('q_bin')['correct'].mean()
    
    acc_by_len.plot(kind='bar', ax=ax, color='#3498db', alpha=0.8)
    ax.set_xlabel('Question Length (chars)')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'{label} — Accuracy by Question Length')
    ax.tick_params(axis='x', rotation=30)
    baseline = 0.25
    ax.axhline(baseline, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 11. LAB Analysis: Temporal Leakage Check

In [ ]:
for stage in STAGES:
    df = dfs[stage]
    for lab_task in ['LAB', 'LAB2000']:
        subset = df[df['task'] == lab_task]
        if len(subset) == 0:
            continue
        acc = subset['correct'].mean()
        conf_right = subset[subset['correct']]['confidence'].median() if subset['correct'].any() else 0
        conf_wrong = subset[~subset['correct']]['confidence'].median() if (~subset['correct']).any() else 0
        verdict = 'OK (no leakage)' if acc < 0.28 else 'WARNING: possible leakage' if acc < 0.32 else 'LEAKING'
        print(f'{stage} | {lab_task}: {acc:.1%} ({verdict}) | conf_right={conf_right:.2f} conf_wrong={conf_wrong:.2f}')